# S13 검색 단계 종료 test 1회 결과

`test` 분할로 세 설정(S5 베이스라인, EXP-014, EXP-017 기준)을 한 번 채점한다(`docs/slices/13-test-run.md`). 같은 test 질의에서 짝지어 비교해 dev 누적 이득이 test에서 얼마나 유지되는지 본다(D-35). test 결과로 기준 설정을 바꾸지 않는다(사용자 결정 2026-10-19). 질의 원문은 출력하지 않는다(D-31).

In [1]:
from pathlib import Path

import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import gold_ranks, load_cfg, load_gold, metric_table, min_docs, per_query, read_run, sha256

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data" / "runs"
CONFIGS = {
    "base": "kure-v1-fixed-512-64",
    "exp014": "snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20",
    "exp017": "snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20-ce",
}
PAIRS = [("base", "exp017"), ("exp014", "exp017"), ("base", "exp014")]
LAYERS = ["dev-full", "test"]

In [3]:
cfg = load_cfg()
ev = cfg.eval
bs = ev.bootstrap
n_min = min_docs(ev.recall_ks, ev.mrr_k)


def layer_scores(layer):
    gold = load_gold(cfg, layer, allow_test=layer == "test")
    return gold, {name: per_query(gold_ranks(read_run(RUNS / f"{prefix}-{layer}.csv"), gold, n_min),
                                  ev.recall_ks, ev.mrr_k)
                  for name, prefix in CONFIGS.items()}


def paired(gold, scores, a, b):
    rows = {}
    for m in scores[a].columns:
        diff = (scores[b][m] - scores[a][m]).to_numpy()
        rows[m] = paired_bootstrap(diff, gold[bs.unit].to_numpy(), bs.n_resamples, bs.seed, bs.alpha)
    return rows

## 순위 파일

In [4]:
pd.DataFrame({layer: {name: sha256(RUNS / f"{p}-{layer}.csv")[:16] for name, p in CONFIGS.items()}
              for layer in LAYERS})

,dev-full,test
base,f15688433784bd16,399514b2fc6c8cbf
exp014,084bc83e39a59ddc,85e152ed830251f2
exp017,c6842e8f2891db88,d905a3d377af2b20


## 설정별 지표

In [5]:
data = {layer: layer_scores(layer) for layer in LAYERS}
pd.DataFrame({(layer, name): s.mean() for layer, (g, sc) in data.items() for name, s in sc.items()}).T.round(4)

recall@1  recall@5  recall@10  mrr@10
dev-full base      0.7062    0.8584     0.8909  0.7707
         exp014    0.8372    0.9413     0.9615  0.8834
         exp017    0.9055    0.9566     0.9665  0.9282
test     base      0.7090    0.8525     0.8904  0.7724
         exp014    0.8338    0.9351     0.9574  0.8787
         exp017    0.9104    0.9577     0.9657  0.9308

## 짝 비교

회의 단위 paired bootstrap(`configs/eval`의 반복 수·seed·유의수준, `rag.eval.compare`와 같은 함수). 차이는 %p다.

In [6]:
rows = []
for layer, (gold, sc) in data.items():
    for a, b in PAIRS:
        for m, (pt, lo, hi) in paired(gold, sc, a, b).items():
            rows.append({"layer": layer, "pair": f"{a}→{b}", "metric": m,
                         "diff": pt * 100, "lo": lo * 100, "hi": hi * 100})
diffs = pd.DataFrame(rows)
diffs.pivot_table(index=["pair", "metric"], columns="layer", values=["diff", "lo", "hi"]).round(2)

diff              hi              lo       
layer                   dev-full   test dev-full   test dev-full   test
pair          metric                                                   
base→exp014   mrr@10       11.27  10.63    12.50  11.86    10.09   9.39
              recall@1     13.10  12.49    14.67  14.12    11.55  10.88
              recall@10     7.06   6.69     8.16   7.78     6.02   5.60
              recall@5      8.29   8.26     9.52   9.51     7.12   7.01
base→exp017   mrr@10       15.75  15.85    17.17  17.22    14.37  14.47
              recall@1     19.93  20.15    21.72  21.85    18.14  18.44
              recall@10     7.56   7.53     8.65   8.68     6.49   6.36
              recall@5      9.81  10.52    11.14  11.88     8.53   9.14
exp014→exp017 mrr@10        4.49   5.22     5.27   5.95     3.70   4.49
              recall@1      6.83   7.66     7.99   8.71     5.65   6.62
              recall@10     0.50   0.83     0.93   1.26     0.09   0.41
              recall@5      1.53   2.26     2.15   2.98     0.91   1.57

## dev 이득의 test 유지 (D-35)

Recall@5 차이의 test/dev 비율. 1이면 dev 이득이 그대로 유지된 것이다.

In [7]:
r5 = diffs[diffs["metric"] == ev.primary].pivot_table(index="pair", columns="layer", values="diff")
r5["test/dev"] = r5["test"] / r5["dev-full"]
r5.round(3)

layer,dev-full,test,test/dev
pair,,,
base→exp014,8.289,8.258,0.996
base→exp017,9.814,10.523,1.072
exp014→exp017,1.525,2.264,1.485


## 층별 (EXP-017, test)

In [8]:
gold_t, sc_t = data["test"]
metric_table(sc_t["exp017"], gold_t, ev.breakdowns).round(4)

,group,value,n,recall@1,recall@5,recall@10,mrr@10
0,all,all,3003,0.9104,0.9577,0.9657,0.9308
1,meeting_name,국정감사,1215,0.9053,0.9556,0.9679,0.9273
2,meeting_name,본회의,99,0.8990,0.9596,0.9697,0.9204
3,meeting_name,소위원회,1207,0.9196,0.9602,0.9644,0.9369
4,meeting_name,예산결산특별위원회,158,0.8481,0.9177,0.9304,0.8810
5,meeting_name,특별위원회,324,0.9290,0.9753,0.9784,0.9488
6,qna_type,단답형,196,0.9082,0.9592,0.9592,0.9307
7,qna_type,추출형,2807,0.9106,0.9576,0.9662,0.9308


## 정리

- 기준(EXP-017) test Recall@5 0.9577(dev-full 0.9566), Recall@1 0.9104, Recall@10 0.9657, MRR@10 0.9308. 층별로는 예산결산특별위원회(158건) Recall@5 0.9177이 가장 낮다
- dev 누적 이득이 test에서 유지됐다. 베이스라인→EXP-017 Recall@5 test +10.52%p [+9.14, +11.88](dev +9.81%p), 리랭커 몫(EXP-014→EXP-017) test +2.26%p [+1.57, +2.98](dev +1.53%p). 베이스라인→EXP-014는 test +8.26%p로 dev(+8.29%p)와 거의 같다. dev를 17번 실험에 반복해 쓴 낙관은 이 지표에서 드러나지 않았다
- test가 확인하지 못하는 것: test 질의도 정답 문서를 보고 생성한 질의라, 어휘 겹침에서 온 이득(특히 리랭커)은 test에서도 그대로 나타난다. 실사용 질의에서의 성능은 이 결과로 알 수 없다
- 단서: test 질의 임베딩은 로컬 CPU float32로 했고, dev 질의 임베딩은 Colab GPU에서 했다(D-07). 세 설정 모두 같은 조건이라 짝 비교에는 영향이 없다. dev와 test의 절대 점수 비교에는 작은 차이가 섞일 수 있다(추정)